Traccia: Utilizzando il codice della lezione come base, modifica l'architettura per implementare una strategia di Fine-Tuning. Sblocca il modello base (vgg_base.trainable=True). Congela manualmente tutti i layer tranne gli ultimi 4. Compila il modello utilizzando un learning rate molto piccolo (10 -5) per evitare di distruggere i pesi pre-addestrati

In [10]:
import tensorflow as tf
from tensorflow.keras import layers, models, applications, optimizers

# 1. CARICAMENTO DEL MODELLO
# include_top=False rimuove i layer densi finali (il classificatore a 1000 classi)
vgg_base = applications.VGG16(weights='imagenet', 
                             include_top=False, #eliminiamo i layer finali
                             input_shape=(224, 224, 3)) #shape di input delle immagini



# 2. CONGELAMENTO DEI PESI (Punto chiave 2)
# Impediamo all'ottimizzatore di modificare i pesi già appresi su ImageNet

# Per prima cosa sblocchiamo l'intera base
vgg_base.trainable = True

# Definiamo quanti layer vogliamo lasciare sbloccati alla fine della rete
fine_tune_at = len(vgg_base.layers) - 4

# Congeliamo tutti i layer fino a fine_tune_at
for layer in vgg_base.layers[:fine_tune_at]:
    layer.trainable = False

#i primi layer sono congelati, gli ultimi 4 sono
#congelare solo i primi layer e lasciare 'liberi' gli ultimi ha senso, soprattutto quando il nostro dataset
#è diverso, perchè i primi layer estraggono caratteristiche più generiche (bordi, texture) che sono utili
#per molti compiti, mentre gli ultimi layer estraggono caratteristiche più specifiche che potrebbero essere più
#rilevanti per il nostro nuovo compito di classificazione, quindi lasciarli addestrabili permette
#al modello di adattarsi meglio al nuovo dataset.

# per verificare i layer
for i, layer in enumerate(vgg_base.layers):
    print(i, layer.name, layer.trainable)

# 3. COSTRUZIONE DEL MODELLO COMPLESSIVO
model = models.Sequential([
    vgg_base,                    # La base convoluzionale "intelligente"
    layers.GlobalAveragePooling2D(), # Trasforma le mappe 2D in un vettore 1D (indicato per l utilizzo con VGG16)
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),         # Regolarizzazione per evitare overfitting
    layers.Dense(64, activation='relu'), # Esempio: 64 nuove categorie
    layers.Dropout(0.5),         # Regolarizzazione per evitare overfitting
    layers.Dense(10, activation='softmax') # Esempio: 10 nuove categorie (deve classificare su 10 categorie)

])
model.summary() #per visualizzare i parametri di addestramento


# 4. COMPILAZIONE con learning rate ridotto
# Usiamo Adam con il learning rate standard perché la base è congelata
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Ispezione dei parametri
model.summary() #per visualizzare i parametri di addestramento

# Verifica visiva dei layer sbloccati
print(f"\nNumero totale di layer in VGG16: {len(vgg_base.layers)}")
print(f"Layer congelati: {fine_tune_at}")
print(f"Layer sbloccati per il fine-tuning: {len(vgg_base.layers) - fine_tune_at}")

for i, layer in enumerate(vgg_base.layers):
    print(f"Layer {i}: {layer.name} | Addestrabile: {layer.trainable}")


0 input_layer_11 False
1 block1_conv1 False
2 block1_conv2 False
3 block1_pool False
4 block2_conv1 False
5 block2_conv2 False
6 block2_pool False
7 block3_conv1 False
8 block3_conv2 False
9 block3_conv3 False
10 block3_pool False
11 block4_conv1 False
12 block4_conv2 False
13 block4_conv3 False
14 block4_pool False
15 block5_conv1 True
16 block5_conv2 True
17 block5_conv3 True
18 block5_pool True


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_4      │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,863,114 (56.70 MB)

 Trainable params: 7,227,850 (27.57 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_4      │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,863,114 (56.70 MB)

 Trainable params: 7,227,850 (27.57 MB)

 Non-trainable params: 7,635,264 (29.13 MB)


Numero totale di layer in VGG16: 19
Layer congelati: 15
Layer sbloccati per il fine-tuning: 4
Layer 0: input_layer_11 | Addestrabile: False
Layer 1: block1_conv1 | Addestrabile: False
Layer 2: block1_conv2 | Addestrabile: False
Layer 3: block1_pool | Addestrabile: False
Layer 4: block2_conv1 | Addestrabile: False
Layer 5: block2_conv2 | Addestrabile: False
Layer 6: block2_pool | Addestrabile: False
Layer 7: block3_conv1 | Addestrabile: False
Layer 8: block3_conv2 | Addestrabile: False
Layer 9: block3_conv3 | Addestrabile: False
Layer 10: block3_pool | Addestrabile: False
Layer 11: block4_conv1 | Addestrabile: False
Layer 12: block4_conv2 | Addestrabile: False
Layer 13: block4_conv3 | Addestrabile: False
Layer 14: block4_pool | Addestrabile: False
Layer 15: block5_conv1 | Addestrabile: True
Layer 16: block5_conv2 | Addestrabile: True
Layer 17: block5_conv3 | Addestrabile: True
Layer 18: block5_pool | Addestrabile: True
